In [7]:
import yfinance as yf
import pandas as pd
from typing import Iterable, Union

TARGET_COLS = [
    "underlying_symbol","quote_date","root","expiration","strike","option_type",
    "open","high","low","close","trade_volume",
    "bid_size_1545","bid_1545","ask_size_1545","ask_1545",
    "underlying_bid_1545","underlying_ask_1545",
    "implied_underlying_price_1545","active_underlying_price_1545",
    "implied_volatility_1545","delta_1545","gamma_1545","theta_1545","vega_1545","rho_1545",
    "bid_size_eod","bid_eod","ask_size_eod","ask_eod",
    "underlying_bid_eod","underlying_ask_eod",
    "vwap","open_interest","delivery_code",
]

def _get_underlying_quote(t: yf.Ticker) -> dict:
    last = pd.NA
    try:
        fi = getattr(t, "fast_info", None)
        if isinstance(fi, dict):
            last = fi.get("last_price", pd.NA)
    except Exception:
        pass

    bid = ask = pd.NA
    try:
        info = t.info or {}
        bid = info.get("bid", pd.NA)
        ask = info.get("ask", pd.NA)
    except Exception:
        pass

    return {"last": last, "bid": bid, "ask": ask}

def _to_schema(
    df: pd.DataFrame,
    *,
    underlying_symbol: str,
    expiration: str,
    option_type: str,  # "C" or "P"
    quote_date: pd.Timestamp,
    underlying_quote: dict,
) -> pd.DataFrame:
    out = pd.DataFrame({
        "underlying_symbol": underlying_symbol,
        "quote_date": quote_date.date(),
        "root": underlying_symbol,
        "expiration": pd.to_datetime(expiration).date(),
        "strike": df["strike"].astype(float),
        "option_type": option_type,

        "open": pd.NA,
        "high": pd.NA,
        "low": pd.NA,
        "close": df.get("lastPrice", pd.NA),

        "trade_volume": df.get("volume", pd.NA),

        "bid_size_1545": pd.NA,
        "bid_1545": pd.NA,
        "ask_size_1545": pd.NA,
        "ask_1545": pd.NA,

        "underlying_bid_1545": pd.NA,
        "underlying_ask_1545": pd.NA,

        "implied_underlying_price_1545": pd.NA,
        "active_underlying_price_1545": underlying_quote.get("last", pd.NA),

        "implied_volatility_1545": df.get("impliedVolatility", pd.NA),

        "delta_1545": pd.NA,
        "gamma_1545": pd.NA,
        "theta_1545": pd.NA,
        "vega_1545": pd.NA,
        "rho_1545": pd.NA,

        "bid_size_eod": pd.NA,
        "bid_eod": df.get("bid", pd.NA),
        "ask_size_eod": pd.NA,
        "ask_eod": df.get("ask", pd.NA),

        "underlying_bid_eod": underlying_quote.get("bid", pd.NA),
        "underlying_ask_eod": underlying_quote.get("ask", pd.NA),

        "vwap": pd.NA,
        "open_interest": df.get("openInterest", pd.NA),
        "delivery_code": "",
    })

    # Copy EOD -> 1545 slots
    out["bid_size_1545"] = out["bid_size_eod"]
    out["bid_1545"]      = out["bid_eod"]
    out["ask_size_1545"] = out["ask_size_eod"]
    out["ask_1545"]      = out["ask_eod"]
    out["underlying_bid_1545"] = out["underlying_bid_eod"]
    out["underlying_ask_1545"] = out["underlying_ask_eod"]

    return out[TARGET_COLS]

def fetch_all_expirations_for_symbol(symbol: str) -> pd.DataFrame:
    t = yf.Ticker(symbol)
    expirations = t.options
    if not expirations:
        return pd.DataFrame(columns=TARGET_COLS)

    quote_date = pd.Timestamp.now(tz="America/New_York")
    underlying_quote = _get_underlying_quote(t)

    frames: list[pd.DataFrame] = []
    for exp in expirations:
        try:
            chain = t.option_chain(exp)

            calls = _to_schema(
                chain.calls,
                underlying_symbol=symbol,
                expiration=exp,
                option_type="C",
                quote_date=quote_date,
                underlying_quote=underlying_quote,
            )
            puts = _to_schema(
                chain.puts,
                underlying_symbol=symbol,
                expiration=exp,
                option_type="P",
                quote_date=quote_date,
                underlying_quote=underlying_quote,
            )

            frames.append(calls)
            frames.append(puts)
        except Exception:
            # Skip this expiration if yfinance errors out for it
            continue

    if not frames:
        return pd.DataFrame(columns=TARGET_COLS)

    return pd.concat(frames, ignore_index=True)

def fetch_all_symbols_all_expirations(symbols: Union[str, Iterable[str]]) -> pd.DataFrame:
    if isinstance(symbols, str):
        symbols = [symbols]
    else:
        symbols = list(symbols)

    all_frames: list[pd.DataFrame] = []
    for sym in symbols:
        try:
            all_frames.append(fetch_all_expirations_for_symbol(sym))
        except Exception:
            # Skip the symbol if yfinance errors out for it
            continue

    if not all_frames:
        return pd.DataFrame(columns=TARGET_COLS)

    return pd.concat(all_frames, ignore_index=True)

# Example:



In [8]:
df = fetch_all_symbols_all_expirations(["NVDA", "LLY", "PLTR", "TSLA", "AMZN", "AAPL"])
# df = fetch_all_symbols_all_expirations(["NVDA", "LLY",])



In [9]:
df.head()

,underlying_symbol,quote_date,root,expiration,strike,option_type,open,high,low,close,...,rho_1545,bid_size_eod,bid_eod,ask_size_eod,ask_eod,underlying_bid_eod,underlying_ask_eod,vwap,open_interest,delivery_code
0,NVDA,2026-01-29,NVDA,2026-01-30,50.0,C,<NA>,<NA>,<NA>,141.95,...,NaN,NaN,139.80,NaN,145.10,181.9,201.11,NaN,22.0,
1,NVDA,2026-01-29,NVDA,2026-01-30,55.0,C,<NA>,<NA>,<NA>,131.48,...,NaN,NaN,136.00,NaN,138.45,181.9,201.11,NaN,12.0,
2,NVDA,2026-01-29,NVDA,2026-01-30,60.0,C,<NA>,<NA>,<NA>,130.33,...,NaN,NaN,129.65,NaN,135.45,181.9,201.11,NaN,12.0,
3,NVDA,2026-01-29,NVDA,2026-01-30,65.0,C,<NA>,<NA>,<NA>,126.32,...,NaN,NaN,124.70,NaN,130.50,181.9,201.11,NaN,3.0,
4,NVDA,2026-01-29,NVDA,2026-01-30,70.0,C,<NA>,<NA>,<NA>,121.34,...,NaN,NaN,119.95,NaN,125.30,181.9,201.11,NaN,26.0,


In [10]:

print(df["expiration"].value_counts().head())

expiration
2026-06-18    1614
2026-12-18    1607
2026-02-20    1059
2026-03-20     993
2027-01-15     966
Name: count, dtype: int64


In [11]:
# df.to_csv('./data/sample_NVDA.csv', index=False)

In [12]:
from __future__ import annotations

import io
import re
from dataclasses import dataclass
from datetime import date, datetime
from typing import Optional, Union

import polars as pl


def _gcs_client():
    try:
        from google.cloud import storage
    except ImportError as exc:  # pragma: no cover - optional dependency
        raise ImportError(
            "google-cloud-storage is required for GCS support. "
            "Install it with `pip install google-cloud-storage`"
        ) from exc
    return storage.Client()


@dataclass
class GCSClosesDataLoader:
    """Uploads daily closes CSVs to GCS using closes-YYYY-MM-DD.csv naming.

    This matches the convention used by GCSClosesDataSource.
    """

    bucket: str
    prefix: str = "closes-"
    extension: str = ".csv"

    def blob_name_for_date(self, d: date) -> str:
        return f"{self.prefix}{d:%Y-%m-%d}{self.extension}"

    def _parse_date_from_blob(self, blob_name: str) -> Optional[date]:
        pattern = rf"^{re.escape(self.prefix)}(\d{{4}}-\d{{2}}-\d{{2}}){re.escape(self.extension)}$"
        m = re.match(pattern, blob_name)
        if not m:
            return None
        return datetime.strptime(m.group(1), "%Y-%m-%d").date()

    def _ensure_valid_date(self, d: Union[date, str]) -> date:
        if isinstance(d, date):
            return d
        # allow "YYYY-MM-DD"
        return datetime.strptime(d, "%Y-%m-%d").date()

    def upload_bytes(
        self,
        csv_bytes: bytes,
        d: Union[date, str],
        *,
        content_type: str = "text/csv",
        overwrite: bool = False,
    ) -> str:
        """Upload CSV bytes to gs://bucket/closes-YYYY-MM-DD.csv; returns blob name."""
        d = self._ensure_valid_date(d)
        blob_name = self.blob_name_for_date(d)

        client = _gcs_client()
        bucket = client.bucket(self.bucket)
        blob = bucket.blob(blob_name)

        if not overwrite and blob.exists(client=client):
            raise FileExistsError(
                f"Refusing to overwrite existing object: gs://{self.bucket}/{blob_name}"
            )

        blob.upload_from_string(csv_bytes, content_type=content_type)

        # sanity check: object name encodes same date
        parsed = self._parse_date_from_blob(blob_name)
        if parsed != d:
            raise RuntimeError(
                f"Upload succeeded but filename/date mismatch: {blob_name} encodes {parsed}, expected {d}"
            )

        return blob_name

    def upload_text(
        self,
        csv_text: str,
        d: Union[date, str],
        *,
        content_type: str = "text/csv",
        overwrite: bool = False,
        encoding: str = "utf-8",
    ) -> str:
        return self.upload_bytes(
            csv_text.encode(encoding),
            d,
            content_type=content_type,
            overwrite=overwrite,
        )

    def upload_df(
        self,
        df: pl.DataFrame,
        d: Union[date, str],
        *,
        overwrite: bool = False,
    ) -> str:
        """Serialize a Polars DataFrame to CSV and upload."""
        buf = io.BytesIO()
        df.write_csv(buf)
        return self.upload_bytes(buf.getvalue(), d, overwrite=overwrite)

    def upload_file(
        self,
        path: str,
        d: Union[date, str],
        *,
        content_type: str = "text/csv",
        overwrite: bool = False,
    ) -> str:
        """Upload a local CSV file as closes-YYYY-MM-DD.csv (renames on upload)."""
        d = self._ensure_valid_date(d)
        blob_name = self.blob_name_for_date(d)

        client = _gcs_client()
        bucket = client.bucket(self.bucket)
        blob = bucket.blob(blob_name)

        if not overwrite and blob.exists(client=client):
            raise FileExistsError(
                f"Refusing to overwrite existing object: gs://{self.bucket}/{blob_name}"
            )

        blob.upload_from_filename(path, content_type=content_type)
        return blob_name


In [ ]:
loader = GCSClosesDataLoader(bucket="ztrade-yesterday-closes")
from datetime import datetime, timedelta
yest_date = (datetime.now().date() - timedelta(days=)).strftime("%Y-%m-%d")

df = pl.from_pandas(df)
blob_name = loader.upload_df(df, yest_date)
print(blob_name)  # closes-2026-01-12.csv


closes-2026-01-29.csv
